# ДЗ1. Дополнительная задача: Толстой и Достоевский

В этом ноутбуке решается дополнительная задача про сравнение словарного роста у Льва Толстого и Федора Достоевского.

Здесь мы не просто считаем статистики, а последовательно отвечаем на три вопроса: как растет словарь в исходных текстах, что меняется после случайного перемешивания токенов и какие слова оказываются наиболее характерными для каждого автора.

Что мы хотим получить:
- собрать тексты `Войны и мира` и `Братьев Карамазовых` из источников задания;
- подготовить чистый текст для анализа;
- сравнить рост словаря в исходных и случайно перемешанных версиях;
- выделить наиболее характерные слова каждого автора.


## 1. Источники данных

Используем страницы `az.lib.ru`, на которые указывает задание.

Для Толстого берем 4 тома `Войны и мира`, для Достоевского берем 4 части `Братьев Карамазовых`. Такой выбор не покрывает всего творчества авторов, поэтому выводы ниже стоит трактовать как выводы по выбранным романам, а не как абсолютный портрет всего авторского стиля.


In [ ]:
from pathlib import Path
from collections import Counter
import random
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from bs4 import BeautifulSoup

plt.style.use("ggplot")
pd.set_option("display.max_colwidth", 200)


In [ ]:
DATA_DIR = Path("additional_data/authors")
DATA_DIR.mkdir(parents=True, exist_ok=True)

TOLSTOY_URLS = {
    "vol1": "http://az.lib.ru/t/tolstoj_lew_nikolaewich/text_0040.shtml",
    "vol2": "http://az.lib.ru/t/tolstoj_lew_nikolaewich/text_0050.shtml",
    "vol3": "http://az.lib.ru/t/tolstoj_lew_nikolaewich/text_0060.shtml",
    "vol4": "http://az.lib.ru/t/tolstoj_lew_nikolaewich/text_0070.shtml",
}

DOSTOEVSKY_URLS = {
    "part1": "http://az.lib.ru/d/dostoewskij_f_m/text_0100.shtml",
    "part2": "http://az.lib.ru/d/dostoewskij_f_m/text_0110.shtml",
    "part3": "http://az.lib.ru/d/dostoewskij_f_m/text_0120.shtml",
    "part4": "http://az.lib.ru/d/dostoewskij_f_m/text_0130.shtml",
}


## 2. Загрузка HTML-страниц

Если файлов еще нет локально, скачиваем их в рабочую папку. Это делает ноутбук воспроизводимым и избавляет от ручной подготовки данных.


In [ ]:
def download_file(url, target_path):
    if target_path.exists():
        return target_path
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    target_path.write_bytes(response.content)
    return target_path


In [ ]:
tolstoy_paths = {
    name: download_file(url, DATA_DIR / f"tolstoy_{name}.shtml")
    for name, url in TOLSTOY_URLS.items()
}

dostoevsky_paths = {
    name: download_file(url, DATA_DIR / f"dostoevsky_{name}.shtml")
    for name, url in DOSTOEVSKY_URLS.items()
}

tolstoy_paths, dostoevsky_paths


## 3. Очистка художественного текста

Страницы `lib.ru` содержат не только роман, но и служебную обвязку сайта. Поэтому мы:
- декодируем HTML как `cp1251`;
- извлекаем текст без тегов;
- берем фрагмент от маркера начала романа до последнего служебного блока `Комментарии:`.


In [ ]:
def extract_book_text(html_path, start_pattern, min_start_index=40):
    html = html_path.read_bytes().decode("cp1251", errors="ignore")
    text = BeautifulSoup(html, "html.parser").get_text("\n")
    lines = [line.strip() for line in text.splitlines() if line.strip()]

    start_idx = next(i for i, line in enumerate(lines) if i >= min_start_index and re.search(start_pattern, line))
    end_candidates = [i for i, line in enumerate(lines) if line.startswith("Комментарии:")]
    end_idx = end_candidates[-1]

    book_lines = lines[start_idx:end_idx]
    return "\n".join(book_lines)


In [ ]:
tolstoy_text = "\n".join(
    extract_book_text(path, r"^ТОМ [А-ЯЁ]+$|^ЧАСТЬ [А-ЯЁ]+\.$")
    for _, path in tolstoy_paths.items()
)

dostoevsky_text = "\n".join(
    extract_book_text(path, r"^ОТ АВТОРА\.$|^ЧАСТЬ [А-ЯЁ]+\.$|^КНИГА [А-ЯЁ]+\.$")
    for _, path in dostoevsky_paths.items()
)

print(len(tolstoy_text), len(dostoevsky_text))


In [ ]:
print(tolstoy_text[:2000])


In [ ]:
print(dostoevsky_text[:2000])


## 4. Токенизация для анализа словаря

Для этой задачи достаточно простой токенизации по русским буквам. Мы приводим все к нижнему регистру и выбрасываем числа и пунктуацию, чтобы сравнение фокусировалось именно на словаре.


In [ ]:
TOKEN_PATTERN = re.compile(r"[а-яё]+", flags=re.IGNORECASE)

def tokenize_russian(text):
    return [token.lower() for token in TOKEN_PATTERN.findall(text)]


In [ ]:
tolstoy_tokens = tokenize_russian(tolstoy_text)
dostoevsky_tokens = tokenize_russian(dostoevsky_text)

authors_stats_df = pd.DataFrame(
    [
        {
            "Автор": "Толстой",
            "Число токенов": len(tolstoy_tokens),
            "Число уникальных токенов": len(set(tolstoy_tokens)),
        },
        {
            "Автор": "Достоевский",
            "Число токенов": len(dostoevsky_tokens),
            "Число уникальных токенов": len(set(dostoevsky_tokens)),
        },
    ]
)
authors_stats_df


## 5. Рост словаря в исходных текстах

Здесь смотрим, как растет число уникальных слов по мере чтения романа. Это тот же тип анализа, что и в основной части, только теперь мы сравниваем не корпус документов, а два больших художественных текста.


In [ ]:
def compute_vocab_growth(tokens, step=5000):
    seen = set()
    xs = []
    ys = []

    for i, token in enumerate(tokens, start=1):
        seen.add(token)
        if i % step == 0:
            xs.append(i)
            ys.append(len(seen))

    if not xs or xs[-1] != len(tokens):
        xs.append(len(tokens))
        ys.append(len(seen))

    return pd.DataFrame({"tokens_seen": xs, "vocab_size": ys})


In [ ]:
tolstoy_growth_df = compute_vocab_growth(tolstoy_tokens)
dostoevsky_growth_df = compute_vocab_growth(dostoevsky_tokens)


In [ ]:
plt.figure(figsize=(10, 6))
plt.loglog(tolstoy_growth_df["tokens_seen"], tolstoy_growth_df["vocab_size"], label="Толстой")
plt.loglog(dostoevsky_growth_df["tokens_seen"], dostoevsky_growth_df["vocab_size"], label="Достоевский")
plt.title("Рост словаря в исходных текстах")
plt.xlabel("Число просмотренных токенов")
plt.ylabel("Размер словаря")
plt.legend()
plt.grid(True, which="both", linestyle="--", alpha=0.5)
plt.show()


## 6. Рост словаря в случайно перемешанных версиях

Теперь повторяем тот же анализ, но предварительно случайно перемешиваем токены. Это позволяет проверить, насколько исходный порядок слов влияет на динамику роста словаря.


In [ ]:
rng = random.Random(42)

tolstoy_tokens_shuffled = tolstoy_tokens.copy()
dostoevsky_tokens_shuffled = dostoevsky_tokens.copy()

rng.shuffle(tolstoy_tokens_shuffled)
rng.shuffle(dostoevsky_tokens_shuffled)


In [ ]:
tolstoy_growth_shuffled_df = compute_vocab_growth(tolstoy_tokens_shuffled)
dostoevsky_growth_shuffled_df = compute_vocab_growth(dostoevsky_tokens_shuffled)


In [ ]:
plt.figure(figsize=(10, 6))
plt.loglog(tolstoy_growth_df["tokens_seen"], tolstoy_growth_df["vocab_size"], label="Толстой: исходный")
plt.loglog(tolstoy_growth_shuffled_df["tokens_seen"], tolstoy_growth_shuffled_df["vocab_size"], label="Толстой: shuffled")
plt.loglog(dostoevsky_growth_df["tokens_seen"], dostoevsky_growth_df["vocab_size"], label="Достоевский: исходный")
plt.loglog(dostoevsky_growth_shuffled_df["tokens_seen"], dostoevsky_growth_shuffled_df["vocab_size"], label="Достоевский: shuffled")
plt.title("Рост словаря: исходные и shuffled-версии")
plt.xlabel("Число просмотренных токенов")
plt.ylabel("Размер словаря")
plt.legend()
plt.grid(True, which="both", linestyle="--", alpha=0.5)
plt.show()


Если shuffled-кривая заметно отличается от исходной, это означает, что распределение слов по тексту неслучайно и порядок появления слов влияет на скорость открытия новых типов.


## 7. Наиболее характерные слова авторов

Чтобы найти характерные слова, сравним относительные частоты слов у двух авторов и посчитаем сглаженное log-odds соотношение. Чем выше score, тем сильнее слово тяготеет к одному автору по сравнению с другим.


In [ ]:
def characteristic_words(tokens_a, tokens_b, top_n=20, min_count=10):
    counts_a = Counter(tokens_a)
    counts_b = Counter(tokens_b)
    vocab = set(counts_a) | set(counts_b)
    total_a = sum(counts_a.values())
    total_b = sum(counts_b.values())
    alpha = 1
    noise_tokens = {"стр", "строка", "изд", "чт", "сноске", "гл", "ч"}

    rows = []
    for token in vocab:
        if token in noise_tokens:
            continue
        count_a = counts_a[token]
        count_b = counts_b[token]
        if count_a + count_b < min_count:
            continue
        score = np.log((count_a + alpha) / (total_a + alpha * len(vocab))) - np.log((count_b + alpha) / (total_b + alpha * len(vocab)))
        rows.append({"token": token, "score": score, "count_a": count_a, "count_b": count_b})

    df = pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)
    return df.head(top_n), df.tail(top_n).iloc[::-1].reset_index(drop=True)


In [ ]:
tolstoy_top_df, dostoevsky_top_df = characteristic_words(tolstoy_tokens, dostoevsky_tokens)

tolstoy_top_df


In [ ]:
dostoevsky_top_df


In [ ]:
plt.figure(figsize=(10, 6))
plot_df = tolstoy_top_df.iloc[::-1]
plt.barh(plot_df["token"], plot_df["score"], color="#4C78A8")
plt.title("Наиболее характерные слова Толстого")
plt.xlabel("Log-odds score")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
plot_df = dostoevsky_top_df.iloc[::-1]
plt.barh(plot_df["token"], -plot_df["score"], color="#E45756")
plt.title("Наиболее характерные слова Достоевского")
plt.xlabel("Log-odds score")
plt.tight_layout()
plt.show()


## 8. Итоговые выводы

По выбранным текстам у Толстого получилось `531 443` токена и `52 075` уникальных слов, у Достоевского `297 011` токенов и `35 929` уникальных слов. В абсолютном размере словарь Толстого больше на `16 146` слов, но относительная лексическая насыщенность по `type-token ratio` выше у Достоевского: `0.1210` против `0.0980`.

Сравнение исходных и shuffled-версий показывает, что порядок слов важен для динамики роста словаря. На отметке около `100 000` токенов перемешивание дает словарь больше на `1663` слова у Толстого и на `547` слов у Достоевского. Максимальный разрыв между кривыми достигает `1819` и `937` слов соответственно. Это означает, что в исходных романах новые слова появляются неравномерно, блоками, связанными с персонажами и сюжетными линиями.

После очистки технического шума наиболее характерные слова Толстого выглядят содержательно: `пьер`, `пьера`, `ростов`, `княжна`, `князь`, `граф`, `кутузов`, `денисов`, `графиня`, `французов`. У Достоевского выделяются `алеша`, `митя`, `смердяков`, `грушенька`, `старец`, `воскликнул`, `дескать`, `карамазов`, `федоровича`, `ракитин`. Эти списки хорошо отражают не только авторский стиль, но и тематику конкретных романов, поэтому в выводе стоит честно отметить влияние выбранных произведений на результат.
